# Multi-Label Baseline Classifier (LinearSVC)

**Purpose**: Establish a strong multi-label classical ML baseline for all 41 CUAD
legal clause categories before transformer fine-tuning.

**Pipeline**: TF-IDF → OneVsRestClassifier(LinearSVC) → 41-label evaluation

**Critical**: Uses contract-level train/test splitting to prevent data leakage.

---
## Section 1 — Imports & Configuration

In [1]:
import json
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    classification_report,
    multilabel_confusion_matrix,
)

# ==========================================
# CONFIGURATION
# ==========================================

RANDOM_SEED = 42
TEST_SIZE = 0.2

np.random.seed(RANDOM_SEED)

# Paths
BASE_DIR = r"C:\Users\chari\Desktop\Contract_Intelligence_AI"
DATASET_PATH = os.path.join(BASE_DIR, "data", "processed", "multi_label_clause_dataset.csv")
MODEL_DIR = os.path.join(BASE_DIR, "models", "multi_label_svm_classifier")

os.makedirs(MODEL_DIR, exist_ok=True)

print("Configuration ready.")
print(f"  Dataset:   {DATASET_PATH}")
print(f"  Model dir: {MODEL_DIR}")
print(f"  Seed:      {RANDOM_SEED}")
print(f"  Test size: {TEST_SIZE}")

Configuration ready.
  Dataset:   C:\Users\chari\Desktop\Contract_Intelligence_AI\data\processed\multi_label_clause_dataset.csv
  Model dir: C:\Users\chari\Desktop\Contract_Intelligence_AI\models\multi_label_svm_classifier
  Seed:      42
  Test size: 0.2


---
## Section 2 — Load Dataset

In [2]:
# ==========================================
# LOAD MULTI-LABEL DATASET
# ==========================================

df = pd.read_csv(DATASET_PATH)

# Identify label columns (everything except text and contract_id)
label_columns = [c for c in df.columns if c not in ["text", "contract_id"]]

print("Dataset Loaded Successfully")
print(f"  Shape:           {df.shape}")
print(f"  Total samples:   {len(df)}")
print(f"  Label columns:   {len(label_columns)}")
print(f"  Unique contracts:{df['contract_id'].nunique()}")
print(f"  Nulls:           {df.isnull().sum().sum()}")

print(f"\nLabel columns ({len(label_columns)}):")
for i, col in enumerate(label_columns):
    pos_count = df[col].sum()
    print(f"  {i+1:2d}. {col:<45s} | positives: {pos_count}")

Dataset Loaded Successfully
  Shape:           (20104, 43)
  Total samples:   20104
  Label columns:   41


  Unique contracts:510
  Nulls:           0

Label columns (41):
   1. Affiliate License-Licensee                    | positives: 59
   2. Affiliate License-Licensor                    | positives: 23
   3. Agreement Date                                | positives: 468
   4. Anti-Assignment                               | positives: 374
   5. Audit Rights                                  | positives: 214
   6. Cap On Liability                              | positives: 275
   7. Change Of Control                             | positives: 121
   8. Competitive Restriction Exception             | positives: 76
   9. Covenant Not To Sue                           | positives: 100
  10. Document Name                                 | positives: 508
  11. Effective Date                                | positives: 387
  12. Exclusivity                                   | positives: 180
  13. Expiration Date                               | positives: 411
  14. Governing Law                      

---
## Section 3 — Contract-Level Train/Test Split

> ⚠️ **CRITICAL**: We split by `contract_id`, not by random rows.
> This ensures snippets from the same contract never appear in both train and test,
> preventing the model from learning document writing style instead of legal semantics.

In [3]:
# ==========================================
# CONTRACT-LEVEL SPLITTING
# ==========================================

def get_contract_level_split(df, test_size=0.2, seed=42):
    """Split dataset at the CONTRACT level to prevent data leakage.

    Ensures no contract appears in both train and test sets.
    """
    rng = np.random.RandomState(seed)

    unique_contracts = df["contract_id"].unique()
    rng.shuffle(unique_contracts)

    split_idx = int(len(unique_contracts) * (1 - test_size))
    train_contracts = set(unique_contracts[:split_idx])
    test_contracts = set(unique_contracts[split_idx:])

    train_df = df[df["contract_id"].isin(train_contracts)].reset_index(drop=True)
    test_df = df[df["contract_id"].isin(test_contracts)].reset_index(drop=True)

    return train_df, test_df, train_contracts, test_contracts


train_df, test_df, train_contracts, test_contracts = get_contract_level_split(
    df, test_size=TEST_SIZE, seed=RANDOM_SEED
)

# ==========================================
# LEAKAGE VERIFICATION
# ==========================================

overlap = train_contracts & test_contracts

print("CONTRACT-LEVEL SPLIT RESULTS")
print("=" * 60)
print(f"  Train contracts: {len(train_contracts)}")
print(f"  Test contracts:  {len(test_contracts)}")
print(f"  Overlap:         {len(overlap)}", end="")
if len(overlap) == 0:
    print(" ✓ ZERO LEAKAGE — safe to proceed")
else:
    print(" ⚠ LEAKAGE DETECTED — STOP!")
print(f"\n  Train snippets:  {len(train_df)} ({100*len(train_df)/len(df):.1f}%)")
print(f"  Test snippets:   {len(test_df)} ({100*len(test_df)/len(df):.1f}%)")

# Show label distribution per split
print(f"\n  Label distribution (train vs test, top 10):")
train_label_sums = train_df[label_columns].sum().sort_values(ascending=False)
test_label_sums = test_df[label_columns].sum()
print(f"  {'Label':<40s} | Train | Test")
print("  " + "-" * 60)
for label in train_label_sums.head(10).index:
    print(f"  {label:<40s} | {int(train_label_sums[label]):>5d} | {int(test_label_sums[label]):>5d}")

CONTRACT-LEVEL SPLIT RESULTS
  Train contracts: 408
  Test contracts:  102
  Overlap:         0 ✓ ZERO LEAKAGE — safe to proceed

  Train snippets:  16073 (79.9%)
  Test snippets:   4031 (20.1%)

  Label distribution (train vs test, top 10):
  Label                                    | Train | Test
  ------------------------------------------------------------
  Document Name                            |   406 |   102
  Parties                                  |   406 |   102
  Agreement Date                           |   374 |    94
  Governing Law                            |   345 |    91
  Expiration Date                          |   330 |    81
  Effective Date                           |   304 |    83
  Anti-Assignment                          |   299 |    75
  Cap On Liability                         |   211 |    64
  License Grant                            |   202 |    52
  Audit Rights                             |   167 |    47


---
## Section 4 — Prepare Features & Targets

In [4]:
# ==========================================
# EXTRACT TEXT AND LABELS
# ==========================================

X_train_text = train_df["text"].values
X_test_text = test_df["text"].values

y_train = train_df[label_columns].values
y_test = test_df[label_columns].values

print("Features and targets prepared.")
print(f"  X_train_text: {X_train_text.shape}")
print(f"  X_test_text:  {X_test_text.shape}")
print(f"  y_train:      {y_train.shape}")
print(f"  y_test:       {y_test.shape}")

# Verify label shapes
assert y_train.shape[1] == 41, f"Expected 41 labels, got {y_train.shape[1]}"
assert y_test.shape[1] == 41, f"Expected 41 labels, got {y_test.shape[1]}"
print(f"\n  ✓ Label matrix verified: {y_train.shape[1]} labels")

Features and targets prepared.
  X_train_text: (16073,)
  X_test_text:  (4031,)
  y_train:      (16073, 41)
  y_test:       (4031, 41)

  ✓ Label matrix verified: 41 labels


---
## Section 5 — TF-IDF Vectorization

In [5]:
# ==========================================
# TF-IDF VECTORIZATION
# ==========================================

tfidf_config = {
    "max_features": 10000,
    "ngram_range": (1, 2),
    "stop_words": "english",
    "min_df": 2,
    "max_df": 0.95,
}

print("TF-IDF Configuration:")
for k, v in tfidf_config.items():
    print(f"  {k}: {v}")

tfidf = TfidfVectorizer(**tfidf_config)

# Fit ONLY on training data to prevent leakage
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

print(f"\nFeature matrices:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  Vocabulary size: {len(tfidf.vocabulary_)}")

TF-IDF Configuration:
  max_features: 10000
  ngram_range: (1, 2)
  stop_words: english
  min_df: 2
  max_df: 0.95



Feature matrices:
  X_train: (16073, 10000)
  X_test:  (4031, 10000)
  Vocabulary size: 10000


---
## Section 6 — Multi-Label Model Training

### Why OneVsRestClassifier + LinearSVC?

- **OneVsRestClassifier** decomposes the multi-label problem into 41 independent
  binary classification tasks — one SVM per label.
- **LinearSVC** is efficient on sparse TF-IDF features and performed well in our
  previous single-label experiments.
- This provides a strong classical ML baseline before transformer fine-tuning.

In [6]:
# ==========================================
# TRAIN MULTI-LABEL LINEARSVC
# ==========================================

print("Training OneVsRestClassifier(LinearSVC)...")
print(f"  Labels: {y_train.shape[1]}")
print(f"  Training samples: {X_train.shape[0]}")
print(f"  Features: {X_train.shape[1]}")

model = OneVsRestClassifier(
    LinearSVC(random_state=RANDOM_SEED),
    n_jobs=-1,
)

model.fit(X_train, y_train)

print(f"\n  ✓ Training complete!")
print(f"  Estimators trained: {len(model.estimators_)}")

Training OneVsRestClassifier(LinearSVC)...
  Labels: 41
  Training samples: 16073
  Features: 10000



  ✓ Training complete!
  Estimators trained: 41


---
## Section 7 — Predictions

In [7]:
# ==========================================
# GENERATE PREDICTIONS
# ==========================================

y_pred = model.predict(X_test)

print("Predictions generated.")
print(f"  y_pred shape: {y_pred.shape}")
print(f"  Total positive predictions: {y_pred.sum()}")
print(f"  Total actual positives:     {y_test.sum()}")
print(f"  Avg predicted labels/sample: {y_pred.sum(axis=1).mean():.2f}")
print(f"  Avg actual labels/sample:    {y_test.sum(axis=1).mean():.2f}")

Predictions generated.
  y_pred shape: (4031, 41)
  Total positive predictions: 769
  Total actual positives:     1334
  Avg predicted labels/sample: 0.19
  Avg actual labels/sample:    0.33


---
## Section 8 — Evaluation Metrics

### Multi-Label Evaluation is Different

In **single-label** classification, a prediction is either right or wrong.

In **multi-label** classification, a prediction can be *partially correct* —
the model might correctly identify 3 out of 5 active labels.

Key metrics:
- **Subset Accuracy** (Exact Match): strictest metric — entire label vector must match
- **Hamming Loss**: fraction of incorrectly predicted labels (lower is better)
- **Micro F1**: aggregates TP/FP/FN across all labels — favors common labels
- **Macro F1**: averages F1 per label — treats all labels equally

### Why Recall Matters Most in Legal AI

In legal document analysis, a **false negative** (missing a risky clause) is far
more dangerous than a **false positive** (flagging a safe clause). A missed liability
cap or termination clause could expose clients to significant legal risk.
Therefore, **recall** is the most critical metric for legal AI systems.

In [8]:
# ==========================================
# COMPUTE OVERALL METRICS
# ==========================================

# Subset accuracy (exact match — strictest metric)
subset_acc = accuracy_score(y_test, y_pred)

# Micro metrics (aggregate across all labels)
micro_precision = precision_score(y_test, y_pred, average="micro", zero_division=0)
micro_recall = recall_score(y_test, y_pred, average="micro", zero_division=0)
micro_f1 = f1_score(y_test, y_pred, average="micro", zero_division=0)

# Macro metrics (average per label — treats all labels equally)
macro_precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
macro_recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

# Hamming loss (fraction of incorrectly predicted labels)
h_loss = hamming_loss(y_test, y_pred)

print("=" * 60)
print("MULTI-LABEL EVALUATION RESULTS")
print("=" * 60)
print(f"\n  Subset Accuracy (Exact Match): {subset_acc:.4f} ({100*subset_acc:.2f}%)")
print(f"  Hamming Loss:                  {h_loss:.4f}")
print(f"\n  Micro Precision: {micro_precision:.4f}")
print(f"  Micro Recall:    {micro_recall:.4f}")
print(f"  Micro F1:        {micro_f1:.4f}")
print(f"\n  Macro Precision: {macro_precision:.4f}")
print(f"  Macro Recall:    {macro_recall:.4f}")
print(f"  Macro F1:        {macro_f1:.4f}")

MULTI-LABEL EVALUATION RESULTS

  Subset Accuracy (Exact Match): 0.7549 (75.49%)
  Hamming Loss:                  0.0072

  Micro Precision: 0.5930
  Micro Recall:    0.3418
  Micro F1:        0.4337

  Macro Precision: 0.4853
  Macro Recall:    0.2280
  Macro F1:        0.2796


---
## Section 9 — Per-Label Performance

In [9]:
# ==========================================
# PER-LABEL METRICS
# ==========================================

per_label_precision = precision_score(y_test, y_pred, average=None, zero_division=0)
per_label_recall = recall_score(y_test, y_pred, average=None, zero_division=0)
per_label_f1 = f1_score(y_test, y_pred, average=None, zero_division=0)

# Build per-label results table
per_label_results = []
for i, label in enumerate(label_columns):
    support = int(y_test[:, i].sum())
    predicted = int(y_pred[:, i].sum())
    per_label_results.append({
        "label": label,
        "precision": round(per_label_precision[i], 4),
        "recall": round(per_label_recall[i], 4),
        "f1": round(per_label_f1[i], 4),
        "support": support,
        "predicted": predicted,
    })

# Sort by F1 descending
per_label_df = pd.DataFrame(per_label_results).sort_values("f1", ascending=False)

print("PER-LABEL PERFORMANCE (sorted by F1)")
print("=" * 90)
print(f"  {'Label':<45s} | {'Prec':>6s} | {'Rec':>6s} | {'F1':>6s} | {'Sup':>5s} | {'Pred':>5s}")
print("  " + "-" * 85)
for _, row in per_label_df.iterrows():
    print(f"  {row['label']:<45s} | {row['precision']:>6.4f} | {row['recall']:>6.4f} | {row['f1']:>6.4f} | {row['support']:>5d} | {row['predicted']:>5d}")

PER-LABEL PERFORMANCE (sorted by F1)
  Label                                         |   Prec |    Rec |     F1 |   Sup |  Pred
  -------------------------------------------------------------------------------------
  Governing Law                                 | 0.7363 | 0.7363 | 0.7363 |    91 |    91
  Anti-Assignment                               | 0.6447 | 0.6533 | 0.6490 |    75 |    76
  Covenant Not To Sue                           | 0.8333 | 0.5000 | 0.6250 |    20 |    12
  Audit Rights                                  | 0.5500 | 0.7021 | 0.6168 |    47 |    60
  Insurance                                     | 0.5526 | 0.6562 | 0.6000 |    32 |    38
  Cap On Liability                              | 0.6957 | 0.5000 | 0.5818 |    64 |    46
  Warranty Duration                             | 1.0000 | 0.3333 | 0.5000 |    15 |     5
  Renewal Term                                  | 0.5556 | 0.4545 | 0.5000 |    33 |    27
  No-Solicit Of Employees                       | 0.8000

---
## Section 10 — False Negative Analysis (Legal Risk)

In [10]:
# ==========================================
# FALSE NEGATIVE ANALYSIS
# ==========================================

print("FALSE NEGATIVE ANALYSIS — Legal Risk Assessment")
print("=" * 60)
print("\nLabels with WORST recall (highest risk of missed clauses):\n")

# Sort by recall ascending to find worst performers
worst_recall = per_label_df.sort_values("recall", ascending=True)

# Compute false negatives per label using multilabel confusion matrix
mcm = multilabel_confusion_matrix(y_test, y_pred)

fn_per_label = {}
for i, label in enumerate(label_columns):
    tn, fp, fn, tp = mcm[i].ravel()
    fn_per_label[label] = int(fn)

# Show top 15 worst recall labels
print(f"  {'Label':<45s} | {'Recall':>7s} | {'FN':>5s} | {'Support':>7s}")
print("  " + "-" * 75)
for _, row in worst_recall.head(15).iterrows():
    fn = fn_per_label[row["label"]]
    marker = " ⚠️" if row["recall"] < 0.5 else ""
    print(f"  {row['label']:<45s} | {row['recall']:>7.4f} | {fn:>5d} | {row['support']:>7d}{marker}")

# Summary
zero_recall = worst_recall[worst_recall["recall"] == 0]
low_recall = worst_recall[worst_recall["recall"] < 0.5]
print(f"\n  Labels with ZERO recall: {len(zero_recall)}")
print(f"  Labels with recall < 0.5: {len(low_recall)}")
total_fn = sum(fn_per_label.values())
print(f"  Total false negatives across all labels: {total_fn}")

FALSE NEGATIVE ANALYSIS — Legal Risk Assessment

Labels with WORST recall (highest risk of missed clauses):

  Label                                         |  Recall |    FN | Support
  ---------------------------------------------------------------------------
  Competitive Restriction Exception             |  0.0000 |    12 |      12 ⚠️
  Affiliate License-Licensee                    |  0.0000 |     8 |       8 ⚠️
  Third Party Beneficiary                       |  0.0000 |     9 |       9 ⚠️
  Price Restrictions                            |  0.0000 |     2 |       2 ⚠️
  Liquidated Damages                            |  0.0000 |    14 |      14 ⚠️
  No-Solicit Of Customers                       |  0.0000 |     8 |       8 ⚠️
  Most Favored Nation                           |  0.0000 |     7 |       7 ⚠️
  Volume Restriction                            |  0.0000 |    17 |      17 ⚠️
  Unlimited/All-You-Can-Eat-License             |  0.0000 |     3 |       3 ⚠️
  Source Code Escrow      

---
## Section 11 — Visualization

In [11]:
# ==========================================
# BAR CHART: Per-label F1 scores
# ==========================================

fig, ax = plt.subplots(figsize=(14, 10))
plot_df = per_label_df.sort_values("f1", ascending=True)
colors = plt.cm.RdYlGn(plot_df["f1"].values)
ax.barh(range(len(plot_df)), plot_df["f1"].values, color=colors, edgecolor="white", linewidth=0.5)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df["label"].values, fontsize=8)
ax.set_xlabel("F1 Score", fontsize=12)
ax.set_title("Per-Label F1 Scores — Multi-Label SVM Baseline (41 Labels)", fontsize=13, fontweight="bold")
ax.set_xlim(0, 1.05)
ax.axvline(x=macro_f1, color="red", linestyle="--", alpha=0.7, label=f"Macro F1 = {macro_f1:.3f}")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, "per_label_f1.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Per-label F1 chart saved.")

Per-label F1 chart saved.


C:\Users\chari\AppData\Local\Temp\ipykernel_22064\1870619984.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# ==========================================
# BAR CHART: Per-label Recall
# ==========================================

fig, ax = plt.subplots(figsize=(14, 10))
plot_df = per_label_df.sort_values("recall", ascending=True)
colors = plt.cm.RdYlGn(plot_df["recall"].values)
ax.barh(range(len(plot_df)), plot_df["recall"].values, color=colors, edgecolor="white", linewidth=0.5)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df["label"].values, fontsize=8)
ax.set_xlabel("Recall", fontsize=12)
ax.set_title("Per-Label Recall — Multi-Label SVM Baseline (Legal Risk View)", fontsize=13, fontweight="bold")
ax.set_xlim(0, 1.05)
ax.axvline(x=macro_recall, color="red", linestyle="--", alpha=0.7, label=f"Macro Recall = {macro_recall:.3f}")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, "per_label_recall.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Per-label recall chart saved.")

Per-label recall chart saved.


C:\Users\chari\AppData\Local\Temp\ipykernel_22064\1773445540.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 12 — Save Outputs

In [13]:
# ==========================================
# SAVE MODEL
# ==========================================

model_path = os.path.join(MODEL_DIR, "svm_model.pkl")
with open(model_path, "wb") as f:
    pickle.dump(model, f)
print(f"✓ Model saved: {model_path}")

# ==========================================
# SAVE TF-IDF VECTORIZER
# ==========================================

tfidf_path = os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl")
with open(tfidf_path, "wb") as f:
    pickle.dump(tfidf, f)
print(f"✓ TF-IDF saved: {tfidf_path}")

✓ Model saved: C:\Users\chari\Desktop\Contract_Intelligence_AI\models\multi_label_svm_classifier\svm_model.pkl
✓ TF-IDF saved: C:\Users\chari\Desktop\Contract_Intelligence_AI\models\multi_label_svm_classifier\tfidf_vectorizer.pkl


In [14]:
# ==========================================
# SAVE LABEL MAPPING
# ==========================================

label_mapping = {
    "label_columns": label_columns,
    "label_to_index": {label: i for i, label in enumerate(label_columns)},
    "index_to_label": {str(i): label for i, label in enumerate(label_columns)},
    "total_labels": len(label_columns),
}

label_map_path = os.path.join(MODEL_DIR, "label_mapping.json")
with open(label_map_path, "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, indent=2, ensure_ascii=False)
print(f"✓ Label mapping saved: {label_map_path}")

✓ Label mapping saved: C:\Users\chari\Desktop\Contract_Intelligence_AI\models\multi_label_svm_classifier\label_mapping.json


In [15]:
# ==========================================
# SAVE METRICS
# ==========================================

per_label_metrics_dict = {}
for _, row in per_label_df.iterrows():
    per_label_metrics_dict[row["label"]] = {
        "precision": row["precision"],
        "recall": row["recall"],
        "f1": row["f1"],
        "support": int(row["support"]),
        "predicted": int(row["predicted"]),
        "false_negatives": fn_per_label[row["label"]],
    }

metrics = {
    "overall": {
        "subset_accuracy": round(subset_acc, 4),
        "hamming_loss": round(h_loss, 4),
    },
    "micro": {
        "precision": round(micro_precision, 4),
        "recall": round(micro_recall, 4),
        "f1": round(micro_f1, 4),
    },
    "macro": {
        "precision": round(macro_precision, 4),
        "recall": round(macro_recall, 4),
        "f1": round(macro_f1, 4),
    },
    "per_label": per_label_metrics_dict,
    "dataset_split": {
        "total_samples": len(df),
        "train_samples": len(train_df),
        "test_samples": len(test_df),
        "train_contracts": len(train_contracts),
        "test_contracts": len(test_contracts),
        "contract_overlap": len(overlap),
        "split_method": "contract-level",
        "test_size": TEST_SIZE,
        "random_seed": RANDOM_SEED,
    },
    "tfidf_config": {
        "max_features": tfidf_config["max_features"],
        "ngram_range": list(tfidf_config["ngram_range"]),
        "stop_words": tfidf_config["stop_words"],
        "min_df": tfidf_config["min_df"],
        "max_df": tfidf_config["max_df"],
        "vocabulary_size": len(tfidf.vocabulary_),
    },
    "model_config": {
        "model": "OneVsRestClassifier(LinearSVC)",
        "base_estimator": "LinearSVC",
        "random_state": RANDOM_SEED,
        "n_estimators": len(model.estimators_),
    },
}

metrics_path = os.path.join(MODEL_DIR, "metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print(f"✓ Metrics saved: {metrics_path}")

✓ Metrics saved: C:\Users\chari\Desktop\Contract_Intelligence_AI\models\multi_label_svm_classifier\metrics.json


---
## Section 13 — Final Summary

In [16]:
# ==========================================
# FINAL BENCHMARK SUMMARY
# ==========================================

print("=" * 60)
print("MULTI-LABEL SVM BASELINE — FINAL SUMMARY")
print("=" * 60)

print(f"\n  Dataset:          {len(df)} samples, {len(label_columns)} labels")
print(f"  Split:            contract-level ({len(train_contracts)} train / {len(test_contracts)} test)")
print(f"  Leakage:          {'NONE' if len(overlap) == 0 else 'DETECTED'}")

print(f"\n  OVERALL METRICS:")
print(f"    Subset Accuracy:  {subset_acc:.4f} ({100*subset_acc:.2f}%)")
print(f"    Hamming Loss:     {h_loss:.4f}")
print(f"    Micro F1:         {micro_f1:.4f}")
print(f"    Macro F1:         {macro_f1:.4f}")
print(f"    Micro Recall:     {micro_recall:.4f}")
print(f"    Macro Recall:     {macro_recall:.4f}")

# Easiest labels (highest F1)
best_labels = per_label_df.head(5)
print(f"\n  TOP 5 EASIEST LABELS (highest F1):")
for _, row in best_labels.iterrows():
    print(f"    {row['label']:<40s} F1={row['f1']:.4f}")

# Hardest labels (lowest F1)
worst_labels = per_label_df.tail(5)
print(f"\n  TOP 5 HARDEST LABELS (lowest F1):")
for _, row in worst_labels.iloc[::-1].iterrows():
    print(f"    {row['label']:<40s} F1={row['f1']:.4f}")

# Prediction summary
labels_with_predictions = (per_label_df["predicted"] > 0).sum()
labels_with_support = (per_label_df["support"] > 0).sum()
print(f"\n  Labels with test support:   {labels_with_support} / {len(label_columns)}")
print(f"  Labels with predictions:   {labels_with_predictions} / {len(label_columns)}")
print(f"  Total false negatives:     {sum(fn_per_label.values())}")

print(f"\n  FILES SAVED:")
print(f"    {model_path}")
print(f"    {tfidf_path}")
print(f"    {label_map_path}")
print(f"    {metrics_path}")

print(f"\n  ✓ Multi-label SVM baseline established")
print(f"  ✓ Ready for comparison with multi-label Legal-BERT fine-tuning")
print(f"    → Next: sigmoid + BCEWithLogitsLoss on same 41 labels")
print("=" * 60)

MULTI-LABEL SVM BASELINE — FINAL SUMMARY

  Dataset:          20104 samples, 41 labels
  Split:            contract-level (408 train / 102 test)
  Leakage:          NONE

  OVERALL METRICS:
    Subset Accuracy:  0.7549 (75.49%)
    Hamming Loss:     0.0072
    Micro F1:         0.4337
    Macro F1:         0.2796
    Micro Recall:     0.3418
    Macro Recall:     0.2280

  TOP 5 EASIEST LABELS (highest F1):
    Governing Law                            F1=0.7363
    Anti-Assignment                          F1=0.6490
    Covenant Not To Sue                      F1=0.6250
    Audit Rights                             F1=0.6168
    Insurance                                F1=0.6000

  TOP 5 HARDEST LABELS (lowest F1):
    Volume Restriction                       F1=0.0000
    Unlimited/All-You-Can-Eat-License        F1=0.0000
    Source Code Escrow                       F1=0.0000
    Third Party Beneficiary                  F1=0.0000
    Price Restrictions                       F1=0.0000

 